# DIMER Workshop: Weather & Earth-System Forecasting with Aurora

**Profile:** `E2E`  
**Mode:** `WORKSHOP`  
**Notebook specification:** `2.1`  
**Status:** Candidate workshop carrier  
**Canonical runtime:** NVIDIA Tesla T4 or equivalent

This standalone workshop demonstrates **short-range global weather forecasting** with Microsoft's Aurora Earth-system foundation model.

The learning workflow is:

> **ERA5 analysis → persistence baseline → frozen Aurora → diagnose resolution/domain shift → bounded LoRA adaptation → validation selection → freeze → independent 6/12/18/24-hour test → spatial interpretation → new-origin forecast → adapter export and fresh reload**

### Core lesson

A foundation model is not automatically useful merely because it is pretrained.

Aurora was trained at **0.25°** resolution. This workshop deliberately uses real ERA5 reanalysis at **1.5°**, six times coarser along each horizontal axis. At that out-of-distribution resolution, the small frozen checkpoint can lose to a very simple persistence forecast. The workshop then asks whether a **540,672-parameter LoRA adaptation** can recover useful short-range skill.

### Weather, not climate projection

This notebook forecasts atmospheric state over **6–24 hours**. It does **not** perform:

- climate projection;
- seasonal climate outlooks;
- long-range climate simulation; or
- climate-change attribution.

Weather forecasting predicts how an atmospheric initial condition evolves over hours to days. Climate projection studies statistical changes in the Earth system over much longer periods and under changing forcings.

### Standalone contract

The default path:

- does not clone a Git repository;
- does not download or import DIMER repository source;
- does not call DIMER workers or APIs;
- requires no token or login;
- uses the upstream `microsoft-aurora` Python package directly;
- pins the Aurora model revision and checkpoint digest;
- obtains real public ERA5/WeatherBench2 data anonymously;
- runs validation, forecasting, LoRA adaptation, evaluation, export, and reload locally.

The notebook is marked **candidate** because its lighter xarray/GCS acquisition path does not yet reproduce the current Aurora pipeline's 57-object per-file digest verification. Exact WeatherBench2 object-pin parity remains a release qualification item.

All measurements are **tutorial/sample-sanity evidence**, not operational forecast verification.

## 0. Learning objectives and resource envelope

By the end of the workshop you should be able to:

1. describe an Aurora atmospheric state;
2. distinguish surface, pressure-level, and static Earth variables;
3. explain why Aurora needs two historical analyses;
4. construct an autoregressive 6–24-hour rollout;
5. implement a persistence baseline;
6. calculate cosine-latitude-weighted global RMSE;
7. read RMSE ratio relative to persistence;
8. recognize resolution/domain shift;
9. perform bounded LoRA adaptation without touching the full backbone;
10. preserve train/validation/test time-window separation;
11. interpret lead-time error growth and spatial error maps;
12. export and reload a SafeTensors adapter; and
13. distinguish an experiment output from an operational weather product.

### Expected resource envelope

Approximate default-path acquisition:

- Aurora small checkpoint: **451 MB**
- WeatherBench2 data actually pulled by xarray for four two-day windows: approximately **200 MB**, subject to store/chunk implementation
- Python packages and caches: additional runtime-dependent storage

The live DIMER Aurora carrier has previously completed an equivalent E2E workflow on a Tesla T4 in roughly five minutes. This candidate workshop must receive its own clean-runtime timing before release.

## 1. Runtime controls

In [ ]:
# @title Workshop controls
USE_BYOD = False                  # @param {type:"boolean"}
BYOD_TRAIN_1_PATH = ""            # @param {type:"string"}
BYOD_TRAIN_2_PATH = ""            # @param {type:"string"}
BYOD_VALIDATION_PATH = ""         # @param {type:"string"}
BYOD_TEST_PATH = ""               # @param {type:"string"}

EPOCHS = 6                        # @param {type:"integer"}
LEARNING_RATE = 0.001             # @param {type:"number"}
TRAINABLE = "lora"                # @param ["lora", "lora+heads"]
MAX_LEAD_STEPS = 4                # @param {type:"integer"}
OUTPUT_DIR = "outputs"            # @param {type:"string"}

SEED = 0

if not 1 <= EPOCHS <= 50:
    raise ValueError("EPOCHS must be in 1..50")
if not 0 < LEARNING_RATE <= 0.1:
    raise ValueError("LEARNING_RATE must be in (0, 0.1]")
if TRAINABLE not in {"lora", "lora+heads"}:
    raise ValueError("TRAINABLE must be 'lora' or 'lora+heads'")
if not 1 <= MAX_LEAD_STEPS <= 8:
    raise ValueError("MAX_LEAD_STEPS must be in 1..8")

from pathlib import Path
OUTPUT_ROOT = Path(OUTPUT_DIR)
for p in [
    OUTPUT_ROOT / "data",
    OUTPUT_ROOT / "baseline",
    OUTPUT_ROOT / "frozen_model",
    OUTPUT_ROOT / "adaptation",
    OUTPUT_ROOT / "frozen",
    OUTPUT_ROOT / "test",
    OUTPUT_ROOT / "future",
    OUTPUT_ROOT / "artifacts" / "aurora_lora",
    OUTPUT_ROOT / "figures",
    OUTPUT_ROOT / "provenance",
]:
    p.mkdir(parents=True, exist_ok=True)

print({
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "trainable": TRAINABLE,
    "max_lead_hours": MAX_LEAD_STEPS * 6,
    "output_root": str(OUTPUT_ROOT.resolve()),
})

## 2. Install one-pass pinned runtime

The model/runtime pins follow the current live DIMER Aurora profile. The workshop additionally installs anonymous Google Cloud/Zarr readers for its simpler standalone WeatherBench2 acquisition path.

No manual kernel restart is part of the intended `Run all` path.

In [ ]:
# @title Install the workshop runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    "torch==2.14.0",
    "microsoft-aurora==2.0.1",
    "timm==1.0.29",
    "einops==0.8.2",
    "xarray==2026.7.0",
    "netCDF4==1.7.4",
    "numcodecs==0.17.0",
    "numpy==2.5.3",
    "safetensors==0.8.0",
    "huggingface-hub==1.32.0",
    "zarr==2.18.7",
    "gcsfs==2026.7.0",
    "matplotlib>=3.9,<3.11",
    "pandas>=2.2,<3.1",
]

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"

if not SKIP_INSTALL:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *PINS],
        check=True,
    )
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import gcsfs
import zarr

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "microsoft-aurora": importlib.metadata.version("microsoft-aurora"),
    "xarray": xr.__version__,
    "numpy": np.__version__,
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("WARNING: T4/CUDA is the canonical workshop runtime; CPU will be substantially slower.")

# 3. Aurora and the Earth-system state

The workshop uses:

**`microsoft/aurora`**  
Revision: `a96afd7ee6d65e3bd2d476f3be798a25a56f2296`

Checkpoint:

**`aurora-0.25-small-pretrained.ckpt`**

This is the approximately **113M-parameter small Aurora checkpoint** that upstream publishes for debugging/testing. It is not the approximately 1.3B-parameter production-scale model.

### Surface variables

| Key | Meaning | Unit |
|---|---|---|
| `2t` | 2 m temperature | K |
| `10u` | 10 m zonal wind | m/s |
| `10v` | 10 m meridional wind | m/s |
| `msl` | mean sea-level pressure | Pa |

### Atmospheric variables on pressure levels

`temperature (t)`, `zonal wind (u)`, `meridional wind (v)`, `specific humidity (q)`, and `geopotential (z)`.

### Pressure levels

`50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000 hPa`

### Static Earth fields

`land-sea mask (lsm)`, `surface geopotential (z)`, and `soil type (slt)`.

Aurora consumes two consecutive six-hourly analyses and predicts the next atmospheric state.

In [ ]:
# @title Immutable model identity and data contract
MODEL_ID = "microsoft/aurora"
MODEL_REVISION = "a96afd7ee6d65e3bd2d476f3be798a25a56f2296"
CHECKPOINT_NAME = "aurora-0.25-small-pretrained.ckpt"
CHECKPOINT_BYTES = 451_339_106
CHECKPOINT_SHA256 = "f80f78de1524a9faba8c9053e4a8ce6a2114ec01cff7f7b4efe9377200d50621"

SURF_VARS = ("2t", "10u", "10v", "msl")
ATMOS_VARS = ("t", "u", "v", "q", "z")
STATIC_VARS = ("lsm", "z", "slt")
LEVELS = (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000)
TIMESTEP_HOURS = 6
HISTORY_STEPS = 2
PATCH_SIZE = 4
LORA_TENSORS_EXPECTED = 80
LORA_PARAMETERS_EXPECTED = 540_672
PARAMETER_COUNT = 112_797_584

UNITS = {
    "2t": "K", "10u": "m/s", "10v": "m/s", "msl": "Pa",
    "t": "K", "u": "m/s", "v": "m/s", "q": "kg/kg", "z": "m²/s²",
}
RANGES = {
    "2t": (150.0, 350.0),
    "10u": (-150.0, 150.0),
    "10v": (-150.0, 150.0),
    "msl": (85_000.0, 110_000.0),
    "t": (150.0, 350.0),
    "u": (-300.0, 300.0),
    "v": (-300.0, 300.0),
    "q": (-1e-3, 0.1),
    "z": (-10_000.0, 250_000.0),
    "lsm": (0.0, 1.0),
    "slt": (0.0, 10.0),
}
LOSS_SCALES = {
    "2t": 10.0, "10u": 5.0, "10v": 5.0, "msl": 1000.0,
    "t": 10.0, "u": 15.0, "v": 15.0, "q": 4e-3, "z": 25_000.0,
}

## 4. Pin and verify the Aurora checkpoint before loading

The workshop asks Hugging Face for the exact immutable revision, checks the expected byte size and SHA-256, and only then lets the upstream package load it through PyTorch's `weights_only=True` path.

Unlike the existing DIMER carrier, this workshop does not perform the additional DIMER one-time SafeTensors conversion for the **base** model. The **adapter** produced by this notebook is SafeTensors. Matching the production carrier's converted-base trust boundary remains a release-hardening item for this candidate workshop.

In [ ]:
# @title Stage and verify the pinned Aurora checkpoint
from huggingface_hub import hf_hub_download
import hashlib

def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

checkpoint_path = Path(hf_hub_download(
    repo_id=MODEL_ID,
    filename=CHECKPOINT_NAME,
    revision=MODEL_REVISION,
))

if checkpoint_path.stat().st_size != CHECKPOINT_BYTES:
    raise ValueError(
        f"Aurora checkpoint size {checkpoint_path.stat().st_size} != pinned {CHECKPOINT_BYTES}"
    )
observed_ckpt_sha = sha256_file(checkpoint_path)
if observed_ckpt_sha != CHECKPOINT_SHA256:
    raise ValueError(f"Aurora checkpoint SHA-256 {observed_ckpt_sha} != pinned {CHECKPOINT_SHA256}")

print({
    "path": str(checkpoint_path),
    "bytes": checkpoint_path.stat().st_size,
    "sha256": observed_ckpt_sha,
    "revision": MODEL_REVISION,
})

# 5. Real ERA5 / WeatherBench2 data

The built-in workshop uses four two-day windows of real ERA5 reanalysis from the public WeatherBench2 1.5° store.

| Role | Window |
|---|---|
| Train 1 | January 2019 |
| Train 2 | July 2019 |
| Validation | April 2020 |
| Test | October 2021 |

Each window contains eight six-hourly analyses on a **121 × 240** global grid.

The windows are separated by whole periods and different years. Individual six-hour analyses are never randomly shuffled across roles.

### Candidate acquisition boundary

The current live DIMER carrier fetches 57 exact Zarr objects with individual byte/digest pins. This workshop uses anonymous xarray/GCS selection against the same public store to keep the standalone reference implementation compact. The selected role/chunk IDs and a post-load dataset digest are recorded, but this does **not** substitute for the 57-object supply-chain check during release qualification.

In [ ]:
# @title WeatherBench2 identities and role windows
from datetime import datetime, timedelta

WB2_BUCKET_PATH = (
    "weatherbench2/datasets/era5/"
    "1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
)
WB2_SOURCE = (
    "gs://" + WB2_BUCKET_PATH
)
WB2_EPOCH = datetime(1959, 1, 1)
WB2_CHUNK_STEPS = 8

SAMPLE_WINDOWS = {
    "train-2019-01": 10957,
    "train-2019-07": 11048,
    "val-2020-04": 11185,
    "test-2021-10": 11459,
}

WB2_SURF = {
    "2t": "2m_temperature",
    "10u": "10m_u_component_of_wind",
    "10v": "10m_v_component_of_wind",
    "msl": "mean_sea_level_pressure",
}
WB2_ATMOS = {
    "t": "temperature",
    "u": "u_component_of_wind",
    "v": "v_component_of_wind",
    "q": "specific_humidity",
    "z": "geopotential",
}
WB2_STATIC = {
    "lsm": "land_sea_mask",
    "z": "geopotential_at_surface",
    "slt": "soil_type",
}

print({
    "source": WB2_SOURCE,
    "window_chunks": SAMPLE_WINDOWS,
    "expected_grid": [121, 240],
    "analyses_per_window": 8,
})

## 6. Load and structurally validate the four windows

The validation layer refuses:

- wrong/missing variables;
- non-finite arrays;
- irregular six-hour spacing;
- the wrong 13-level set;
- implausible physical ranges;
- incompatible grids.

Passing these checks does **not** prove that an atmospheric state is dynamically realistic.

In [ ]:
# @title Data loader and validator
def _transpose_values(da, dims):
    return np.asarray(da.transpose(*dims).values, dtype=np.float32)

def _load_builtin_windows():
    fs = gcsfs.GCSFileSystem(token="anon")
    mapper = fs.get_mapper(WB2_BUCKET_PATH)
    try:
        ds = xr.open_zarr(mapper, consolidated=True)
    except Exception:
        ds = xr.open_zarr(mapper, consolidated=False)

    lat = np.asarray(ds["latitude"].values, dtype=np.float64)
    lon = np.asarray(ds["longitude"].values, dtype=np.float64)
    levels = tuple(int(x) for x in np.asarray(ds["level"].values).tolist())

    windows = {}
    for name, chunk in SAMPLE_WINDOWS.items():
        first = chunk * WB2_CHUNK_STEPS
        sl = slice(first, first + WB2_CHUNK_STEPS)
        times = [
            (WB2_EPOCH + timedelta(hours=TIMESTEP_HOURS * (first + i))).strftime("%Y-%m-%dT%H:%M:%S")
            for i in range(WB2_CHUNK_STEPS)
        ]

        surf = {
            key: _transpose_values(ds[var].isel(time=sl), ("time", "latitude", "longitude"))
            for key, var in WB2_SURF.items()
        }
        atmos = {
            key: _transpose_values(
                ds[var].isel(time=sl).sel(level=list(LEVELS)),
                ("time", "level", "latitude", "longitude"),
            )
            for key, var in WB2_ATMOS.items()
        }
        static = {
            key: _transpose_values(ds[var], ("latitude", "longitude"))
            for key, var in WB2_STATIC.items()
        }
        windows[name] = {
            "name": name,
            "lat": lat.copy(),
            "lon": lon.copy(),
            "levels": list(levels),
            "times": times,
            "surf": surf,
            "atmos": atmos,
            "static": static,
            "source": "ERA5 via WeatherBench2 1.5°",
        }
    return windows

def _load_window_netcdf(path):
    ds = xr.load_dataset(path, engine="netcdf4")
    lat = np.asarray(ds["latitude"].values, dtype=np.float64)
    lon = np.asarray(ds["longitude"].values, dtype=np.float64)
    levels = [int(x) for x in np.asarray(ds["level"].values).tolist()]
    times = [
        str(np.datetime_as_string(t, unit="s"))
        for t in np.asarray(ds["time"].values)
    ]
    surf = {}
    atmos = {}
    static = {}
    for key in SURF_VARS:
        candidates = [f"surf_{key}", key]
        found = next((c for c in candidates if c in ds), None)
        if found is None:
            raise ValueError(f"{path}: missing surface variable {key}")
        surf[key] = np.asarray(ds[found].values, dtype=np.float32)
    for key in ATMOS_VARS:
        candidates = [f"atmos_{key}", key]
        found = next((c for c in candidates if c in ds), None)
        if found is None:
            raise ValueError(f"{path}: missing atmospheric variable {key}")
        atmos[key] = np.asarray(ds[found].values, dtype=np.float32)
    for key in STATIC_VARS:
        candidates = [f"static_{key}", f"{key}_static"]
        found = next((c for c in candidates if c in ds), None)
        if found is None:
            raise ValueError(f"{path}: missing static field {key}")
        static[key] = np.asarray(ds[found].values, dtype=np.float32)
    return {
        "name": Path(path).stem,
        "lat": lat,
        "lon": lon,
        "levels": levels,
        "times": times,
        "surf": surf,
        "atmos": atmos,
        "static": static,
        "source": f"BYOD NetCDF: {path}",
    }

def validate_window(window):
    required = {"name", "lat", "lon", "levels", "times", "surf", "atmos", "static"}
    missing = required - set(window)
    if missing:
        raise ValueError(f"{window.get('name','window')}: missing keys {sorted(missing)}")

    lat = np.asarray(window["lat"], dtype=np.float64)
    lon = np.asarray(window["lon"], dtype=np.float64)
    if lat.ndim != 1 or lon.ndim != 1 or not np.isfinite(lat).all() or not np.isfinite(lon).all():
        raise ValueError("latitude/longitude must be finite 1-D arrays")

    if lat[0] < lat[-1]:
        lat = lat[::-1]
        flip = True
    else:
        flip = False
    if not np.all(np.diff(lat) < 0):
        raise ValueError("latitudes must be strictly decreasing")
    if not np.all(np.diff(lon) > 0):
        raise ValueError("longitudes must be strictly increasing")
    if abs(lat[0] - 90) > 1e-5 or abs(lat[-1] + 90) > 1e-5:
        raise ValueError("latitude grid must span 90 to -90")
    if len(lon) % PATCH_SIZE:
        raise ValueError("longitude count must be divisible by Aurora patch size 4")
    if len(lat) % PATCH_SIZE not in (0, 1):
        raise ValueError("latitude count must be divisible by 4 or one row larger")

    if tuple(int(x) for x in window["levels"]) != LEVELS:
        raise ValueError(f"pressure levels must be exactly {LEVELS}")

    times = [datetime.fromisoformat(str(t)[:19]) for t in window["times"]]
    if not 3 <= len(times) <= 64:
        raise ValueError("window needs 3..64 analyses")
    for a, b in zip(times[:-1], times[1:]):
        if b - a != timedelta(hours=6):
            raise ValueError("analyses must be spaced exactly 6 hours apart")

    h, w = len(lat), len(lon)
    n = len(times)

    def check_array(group, key, expected):
        source = window[group]
        if key not in source:
            raise ValueError(f"{group}: missing {key}")
        arr = np.asarray(source[key], dtype=np.float32)
        if flip:
            arr = arr[..., ::-1, :]
        if arr.shape != expected:
            raise ValueError(f"{group}/{key}: shape {arr.shape}, expected {expected}")
        if not np.isfinite(arr).all():
            raise ValueError(f"{group}/{key}: non-finite values")
        lo, hi = RANGES[key]
        if float(arr.min()) < lo or float(arr.max()) > hi:
            raise ValueError(f"{group}/{key}: values outside plausible range {(lo,hi)}")
        return np.ascontiguousarray(arr)

    surf = {k: check_array("surf", k, (n, h, w)) for k in SURF_VARS}
    atmos = {k: check_array("atmos", k, (n, len(LEVELS), h, w)) for k in ATMOS_VARS}
    static = {k: check_array("static", k, (h, w)) for k in STATIC_VARS}

    return {
        **window,
        "lat": np.ascontiguousarray(lat),
        "lon": np.ascontiguousarray(lon),
        "levels": list(LEVELS),
        "times": [t.strftime("%Y-%m-%dT%H:%M:%S") for t in times],
        "surf": surf,
        "atmos": atmos,
        "static": static,
        "shape": (h, w),
        "n_steps": n,
        "resolution_degrees": float(360 / w),
    }

def window_digest(window):
    checked = validate_window(window)
    h = hashlib.sha256()
    h.update(json.dumps({
        "name": checked["name"],
        "lat": checked["lat"].tolist(),
        "lon": checked["lon"].tolist(),
        "times": checked["times"],
        "levels": checked["levels"],
    }, separators=(",", ":")).encode())
    for group in ("surf", "atmos", "static"):
        for key in sorted(checked[group]):
            h.update(group.encode()); h.update(key.encode())
            h.update(checked[group][key].tobytes())
    return h.hexdigest()

if USE_BYOD:
    required_paths = [
        BYOD_TRAIN_1_PATH, BYOD_TRAIN_2_PATH,
        BYOD_VALIDATION_PATH, BYOD_TEST_PATH,
    ]
    if not all(required_paths):
        raise ValueError(
            "Full BYOD E2E mode requires all four paths: two train windows, one validation, one test"
        )
    raw_windows = {
        "train-1": _load_window_netcdf(BYOD_TRAIN_1_PATH),
        "train-2": _load_window_netcdf(BYOD_TRAIN_2_PATH),
        "validation": _load_window_netcdf(BYOD_VALIDATION_PATH),
        "test": _load_window_netcdf(BYOD_TEST_PATH),
    }
else:
    raw_windows = _load_builtin_windows()

windows = {name: validate_window(w) for name, w in raw_windows.items()}

if USE_BYOD:
    train_windows = [windows["train-1"], windows["train-2"]]
    val_window = windows["validation"]
    test_window = windows["test"]
else:
    train_windows = [windows["train-2019-01"], windows["train-2019-07"]]
    val_window = windows["val-2020-04"]
    test_window = windows["test-2021-10"]

shapes = {tuple(w["shape"]) for w in windows.values()}
if len(shapes) != 1:
    raise ValueError(f"all workshop windows must use one grid; got {shapes}")

dataset_manifest = {
    "source": "BYOD" if USE_BYOD else "ERA5 via WeatherBench2 1.5°",
    "acquisition": (
        "user-supplied NetCDF"
        if USE_BYOD else
        "anonymous xarray/gcsfs selection from public WeatherBench2 Zarr; "
        "per-object DIMER digest parity pending release qualification"
    ),
    "roles": {
        "train": [w["name"] for w in train_windows],
        "validation": val_window["name"],
        "test": test_window["name"],
    },
    "grid": list(next(iter(shapes))),
    "resolution_degrees": train_windows[0]["resolution_degrees"],
    "digests": {name: window_digest(w) for name, w in windows.items()},
}

(OUTPUT_ROOT / "data" / "dataset_manifest.json").write_text(
    json.dumps(dataset_manifest, indent=2), encoding="utf-8"
)

print(json.dumps(dataset_manifest, indent=2))

# 7. Explore the Earth-system state

A weather-model input is a coupled spatial field, not a collection of independent rows.

The following views show:

- 2 m temperature;
- mean sea-level pressure;
- 500 hPa geopotential;
- 850 hPa zonal wind;
- land-sea mask.

The RGB-image intuition from computer vision does not apply: each variable has its own physical units and vertical meaning.

In [ ]:
# @title Visualize selected validation fields
LEVEL_INDEX = {level: i for i, level in enumerate(LEVELS)}
sample = val_window
time_index = 1
extent = [sample["lon"][0], sample["lon"][-1], sample["lat"][-1], sample["lat"][0]]

fields = [
    ("2 m temperature [K]", sample["surf"]["2t"][time_index]),
    ("Mean sea-level pressure [Pa]", sample["surf"]["msl"][time_index]),
    ("500 hPa geopotential [m²/s²]", sample["atmos"]["z"][time_index, LEVEL_INDEX[500]]),
    ("850 hPa zonal wind [m/s]", sample["atmos"]["u"][time_index, LEVEL_INDEX[850]]),
    ("Land-sea mask", sample["static"]["lsm"]),
]

for title, field in fields:
    fig, ax = plt.subplots(figsize=(11, 4))
    im = ax.imshow(field, extent=extent, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()
    safe = title.lower().replace(" ", "_").replace("/", "_")[:40]
    plt.savefig(OUTPUT_ROOT / "figures" / f"state_{safe}.png", dpi=140, bbox_inches="tight")
    plt.show()

# 8. Area weighting on a latitude–longitude grid

A regular latitude–longitude grid does not assign equal physical area to every cell.

Cells near the poles represent much less area than cells near the equator. The canonical global RMSE therefore uses:

\[
w(\phi)=\cos(\phi)
\]

with the latitude weights normalized to unit mean.

**Exercise:** What would happen if every 1.5° grid cell were weighted equally?

In [ ]:
# @title Weighted and unweighted RMSE helpers
def lat_weights(lat):
    w = np.cos(np.deg2rad(np.asarray(lat, dtype=np.float64)))
    return w / w.mean()

def lat_weighted_rmse(pred, truth, lat):
    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if pred.shape != truth.shape:
        raise ValueError(f"prediction {pred.shape} != truth {truth.shape}")
    w = lat_weights(lat)
    w = w[:, None] if pred.ndim == 2 else w[None, :, None]
    return float(np.sqrt(np.mean((pred - truth) ** 2 * w)))

def unweighted_rmse(pred, truth):
    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    return float(np.sqrt(np.mean((pred-truth)**2)))

# A persistence example on validation 2t at +6 h.
origin = 1
height = val_window["shape"][0] - (val_window["shape"][0] % PATCH_SIZE)
truth = val_window["surf"]["2t"][origin+1, :height]
persist = val_window["surf"]["2t"][origin, :height]
lat = val_window["lat"][:height]

print({
    "unweighted_rmse_K": unweighted_rmse(persist, truth),
    "latitude_weighted_rmse_K": lat_weighted_rmse(persist, truth, lat),
})

# 9. Build Aurora from the pinned checkpoint

Two model instances are conceptually important:

1. **frozen Aurora** — base checkpoint only;
2. **adaptable Aurora** — same base checkpoint with zero-initialized LoRA tensors.

Before LoRA training, those two models should represent the same pretrained forecasting function apart from numerical implementation effects.

In [ ]:
# @title Model and Batch helpers
from aurora import AuroraSmallPretrained, Batch, Metadata, rollout

def build_aurora(use_lora):
    torch.manual_seed(SEED)
    model = AuroraSmallPretrained(use_lora=use_lora)
    # The source file was byte/digest verified above. The upstream loader uses
    # torch.load(..., weights_only=True) and adapts historical parameter names.
    model.load_checkpoint_local(str(checkpoint_path), strict=not use_lora)
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    if use_lora:
        names = [n for n, _ in model.named_parameters() if "lora" in n]
        n_params = sum(p.numel() for n, p in model.named_parameters() if "lora" in n)
        if len(names) != LORA_TENSORS_EXPECTED:
            raise ValueError(f"LoRA tensor count {len(names)} != expected {LORA_TENSORS_EXPECTED}")
        if n_params != LORA_PARAMETERS_EXPECTED:
            raise ValueError(f"LoRA parameter count {n_params} != expected {LORA_PARAMETERS_EXPECTED}")
    return model

def make_batch(window, origin):
    if origin < HISTORY_STEPS-1 or origin >= window["n_steps"]:
        raise ValueError(f"origin {origin} outside valid range")
    lo, hi = origin-HISTORY_STEPS+1, origin+1
    return Batch(
        surf_vars={
            k: torch.from_numpy(window["surf"][k][lo:hi][None])
            for k in SURF_VARS
        },
        static_vars={
            k: torch.from_numpy(window["static"][k])
            for k in STATIC_VARS
        },
        atmos_vars={
            k: torch.from_numpy(window["atmos"][k][lo:hi][None])
            for k in ATMOS_VARS
        },
        metadata=Metadata(
            lat=torch.tensor(window["lat"], dtype=torch.float32),
            lon=torch.tensor(window["lon"], dtype=torch.float32),
            time=(datetime.fromisoformat(window["times"][origin]),),
            atmos_levels=LEVELS,
        ),
    )

def forecast_from_origin(model, window, origin, steps):
    batch = make_batch(window, origin)
    with torch.inference_mode():
        preds = [p.to("cpu") for p in rollout(model, batch.to(DEVICE), steps=steps)]
    return preds

adapted_model = build_aurora(use_lora=True)

print({
    "base_parameters": PARAMETER_COUNT,
    "lora_parameters": LORA_PARAMETERS_EXPECTED,
    "lora_fraction_percent": 100 * LORA_PARAMETERS_EXPECTED / PARAMETER_COUNT,
    "device": DEVICE,
})

# 10. Persistence first

Persistence simply carries the latest analysis forward:

\[
\hat y_{t+h}=y_t
\]

At six hours it is often a surprisingly strong baseline. A learned weather model that cannot beat persistence on its deployment domain has not yet demonstrated useful forecast skill.

The workshop therefore calculates persistence **before** reporting Aurora.

In [ ]:
# @title Common forecast evaluator
HEADLINE_LEVEL = 500
REPORTED = (*SURF_VARS, *ATMOS_VARS, "z500")

def forecast_metrics(window, origins, predictions):
    height = window["shape"][0] - (window["shape"][0] % PATCH_SIZE)
    lat = window["lat"][:height]
    level_500 = LEVELS.index(500)
    n_leads = len(predictions[0])
    result = {name: {} for name in REPORTED}

    for k in range(n_leads):
        lead = f"{(k+1)*6}h"
        model_errors = {name: [] for name in REPORTED}
        persistence_errors = {name: [] for name in REPORTED}

        for origin, preds in zip(origins, predictions):
            target_index = origin + k + 1
            for name in SURF_VARS:
                truth = window["surf"][name][target_index, :height]
                model_errors[name].append(
                    lat_weighted_rmse(preds[k]["surf"][name], truth, lat)
                )
                persistence_errors[name].append(
                    lat_weighted_rmse(window["surf"][name][origin, :height], truth, lat)
                )
            for name in ATMOS_VARS:
                truth = window["atmos"][name][target_index, :, :height]
                model_errors[name].append(
                    lat_weighted_rmse(preds[k]["atmos"][name], truth, lat)
                )
                persistence_errors[name].append(
                    lat_weighted_rmse(window["atmos"][name][origin, :, :height], truth, lat)
                )

            truth = window["atmos"]["z"][target_index, level_500, :height]
            model_errors["z500"].append(
                lat_weighted_rmse(preds[k]["atmos"]["z"][level_500], truth, lat)
            )
            persistence_errors["z500"].append(
                lat_weighted_rmse(window["atmos"]["z"][origin, level_500, :height], truth, lat)
            )

        for name in REPORTED:
            model_rmse = float(np.mean(model_errors[name]))
            persistence_rmse = float(np.mean(persistence_errors[name]))
            result[name][lead] = {
                "model": model_rmse,
                "persistence": persistence_rmse,
                "skill": model_rmse / persistence_rmse if persistence_rmse > 0 else np.nan,
            }

    leads = [f"{(k+1)*6}h" for k in range(n_leads)]
    summary = {
        lead: {
            "mean_skill": float(np.mean([
                result[name][lead]["skill"] for name in REPORTED if name != "z500"
            ])),
            "variables_beating_persistence": int(sum(
                result[name][lead]["skill"] < 1
                for name in REPORTED if name != "z500"
            )),
        }
        for lead in leads
    }
    return {
        "n_origins": len(origins),
        "leads": leads,
        "variables": result,
        "summary": summary,
        "metric": "latitude-weighted RMSE; skill = model RMSE / persistence RMSE",
    }

def evaluate_model(model, window, max_steps=4):
    last_origin = window["n_steps"] - 1 - max_steps
    origins = list(range(HISTORY_STEPS-1, last_origin+1))
    if not origins:
        raise ValueError("window too short for requested lead")
    predictions = []
    for origin in origins:
        preds = forecast_from_origin(model, window, origin, max_steps)
        predictions.append([
            {
                "surf": {k: p.surf_vars[k][0,0].numpy() for k in SURF_VARS},
                "atmos": {k: p.atmos_vars[k][0,0].numpy() for k in ATMOS_VARS},
            }
            for p in preds
        ])
    return forecast_metrics(window, origins, predictions)

def persistence_only(window, max_steps=4):
    height = window["shape"][0] - (window["shape"][0] % PATCH_SIZE)
    last_origin = window["n_steps"] - 1 - max_steps
    origins = list(range(HISTORY_STEPS-1, last_origin+1))
    predictions = [
        [
            {
                "surf": {k: window["surf"][k][origin, :height] for k in SURF_VARS},
                "atmos": {k: window["atmos"][k][origin, :, :height] for k in ATMOS_VARS},
            }
            for _ in range(max_steps)
        ]
        for origin in origins
    ]
    scored = forecast_metrics(window, origins, predictions)
    return {
        "leads": scored["leads"],
        "variables": {
            name: {
                lead: {"persistence": values["persistence"]}
                for lead, values in lead_values.items()
            }
            for name, lead_values in scored["variables"].items()
        },
        "baseline": "persistence",
    }

val_persistence = persistence_only(val_window, MAX_LEAD_STEPS)
(OUTPUT_ROOT / "baseline" / "validation_persistence.json").write_text(
    json.dumps(val_persistence, indent=2), encoding="utf-8"
)

print("Validation persistence RMSE:")
for name in ("2t", "msl", "z500"):
    print(name, {
        lead: round(values["persistence"], 4)
        for lead, values in val_persistence["variables"][name].items()
    })

# 11. Frozen Aurora on validation — diagnose the domain shift

The adaptable model currently has zero-initialized LoRA tensors, so it is still the frozen pretrained forecast.

At the tutorial's 1.5° grid, **losing to persistence is an expected and scientifically useful result**. It demonstrates why variable names and units alone are not enough to guarantee transfer: each grid cell and each 4×4 token now represents a much larger physical area than during pretraining.

In [ ]:
# @title Frozen validation forecast
import time

started = time.perf_counter()
frozen_validation = evaluate_model(adapted_model, val_window, MAX_LEAD_STEPS)
frozen_validation_seconds = time.perf_counter() - started

(OUTPUT_ROOT / "frozen_model" / "validation_metrics.json").write_text(
    json.dumps(frozen_validation, indent=2), encoding="utf-8"
)

validation_summary = pd.DataFrame([
    {
        "lead": lead,
        "mean_rmse_ratio_vs_persistence": frozen_validation["summary"][lead]["mean_skill"],
        "variables_beating_persistence": frozen_validation["summary"][lead]["variables_beating_persistence"],
        "of": 9,
    }
    for lead in frozen_validation["leads"]
])
display(validation_summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    [int(x[:-1]) for x in validation_summary["lead"]],
    validation_summary["mean_rmse_ratio_vs_persistence"],
    marker="o",
)
ax.axhline(1.0, linestyle="--")
ax.set_xlabel("Lead time [hours]")
ax.set_ylabel("Mean RMSE ratio vs persistence")
ax.set_title("Frozen Aurora at the 1.5° validation grid")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "figures" / "validation_frozen_skill.png", dpi=150, bbox_inches="tight")
plt.show()

## Interpretation checkpoint

Aurora was trained at 0.25°. At 1.5°:

- horizontal gradients are smoother;
- extremes are reduced by regridding;
- orography is coarser;
- the spatial scale represented by each token changes dramatically.

**Exercise:** Why should a pretrained neural forecaster be expected to notice that change even if the variable names, units, and global coverage remain correct?

# 12. LoRA adaptation

Aurora's attention layers expose rank-8 LoRA parameters.

Canonical adaptation:

- 80 LoRA tensors;
- 540,672 trainable parameters;
- approximately 0.48% of the 113M base model;
- train on January + July 2019;
- validate on April 2020;
- six epochs;
- AdamW;
- one-step (+6 h) training loss;
- lowest validation loss selects the final state.

Epoch 0 — the frozen model — remains eligible. If adaptation does not improve the validation objective, the workshop keeps the frozen policy.

In [ ]:
# @title LoRA adaptation helpers
import copy

def trainable_names(model, mode):
    if mode not in {"lora", "lora+heads"}:
        raise ValueError(mode)
    prefixes = (
        "decoder.surf_heads.",
        "decoder.atmos_heads.",
        "encoder.surf_token_embeds.",
        "encoder.atmos_token_embeds.",
    )
    names = []
    for name, _ in model.named_parameters():
        if "lora" in name or (mode == "lora+heads" and name.startswith(prefixes)):
            names.append(name)
    return names

def target_fields(window, origin):
    height = window["shape"][0] - (window["shape"][0] % PATCH_SIZE)
    return {
        **{
            k: torch.from_numpy(window["surf"][k][origin+1, :height]).to(DEVICE)
            for k in SURF_VARS
        },
        **{
            k: torch.from_numpy(window["atmos"][k][origin+1, :, :height]).to(DEVICE)
            for k in ATMOS_VARS
        },
    }

def scaled_loss(pred, target, scales):
    total = 0.0
    for k in SURF_VARS:
        total = total + torch.mean(((pred.surf_vars[k][0,0] - target[k]) / scales[k])**2)
    for k in ATMOS_VARS:
        total = total + torch.mean(((pred.atmos_vars[k][0,0] - target[k]) / scales[k])**2)
    return total / (len(SURF_VARS) + len(ATMOS_VARS))

def one_step_validation_loss(model, window, scales):
    model.eval()
    values = []
    with torch.inference_mode():
        for origin in range(HISTORY_STEPS-1, window["n_steps"]-1):
            pred = model(make_batch(window, origin).to(DEVICE))
            values.append(float(scaled_loss(pred, target_fields(window, origin), scales)))
    return float(np.mean(values))

def adapt_model(model, train_windows, val_window, epochs, lr, mode, seed):
    torch.manual_seed(seed)
    names = trainable_names(model, mode)
    wanted = set(names)

    for name, param in model.named_parameters():
        param.requires_grad_(name in wanted)

    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)

    if mode == "lora":
        if len(names) != LORA_TENSORS_EXPECTED:
            raise ValueError(f"LoRA tensor count {len(names)} != {LORA_TENSORS_EXPECTED}")
        if n_trainable != LORA_PARAMETERS_EXPECTED:
            raise ValueError(f"LoRA params {n_trainable} != {LORA_PARAMETERS_EXPECTED}")

    optimizer = torch.optim.AdamW(params, lr=lr)
    scales = {
        k: torch.tensor(v, dtype=torch.float32, device=DEVICE)
        for k, v in LOSS_SCALES.items()
    }

    samples = [
        (window, origin)
        for window in train_windows
        for origin in range(HISTORY_STEPS-1, window["n_steps"]-1)
    ]
    generator = torch.Generator().manual_seed(seed)

    initial = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
    best_state = copy.deepcopy(initial)
    best_epoch = 0
    best_val = one_step_validation_loss(model, val_window, scales)
    history = [{
        "epoch": 0,
        "train_loss": None,
        "val_loss": best_val,
        "note": "frozen model / zero LoRA",
    }]

    started = time.perf_counter()
    try:
        for epoch in range(1, epochs+1):
            model.train()
            order = torch.randperm(len(samples), generator=generator).tolist()
            losses = []
            for idx in order:
                window, origin = samples[idx]
                pred = model(make_batch(window, origin).to(DEVICE))
                loss = scaled_loss(pred, target_fields(window, origin), scales)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step()
                losses.append(float(loss.detach()))

            val_loss = one_step_validation_loss(model, val_window, scales)
            entry = {
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "val_loss": val_loss,
            }
            history.append(entry)
            print(entry)

            if val_loss < best_val:
                best_val = val_loss
                best_epoch = epoch
                best_state = {
                    k: v.detach().clone()
                    for k, v in model.state_dict().items()
                    if k in wanted
                }
    except BaseException:
        merged = dict(model.state_dict())
        merged.update(initial)
        model.load_state_dict(merged, strict=True)
        for p in model.parameters():
            p.requires_grad_(False)
        model.eval()
        raise

    merged = dict(model.state_dict())
    merged.update(best_state)
    model.load_state_dict(merged, strict=True)
    for p in model.parameters():
        p.requires_grad_(False)
    model.eval()

    return {
        "trainable": mode,
        "trainable_names": names,
        "n_trainable": n_trainable,
        "n_total": sum(p.numel() for p in model.parameters()),
        "epochs": epochs,
        "best_epoch": best_epoch,
        "learning_rate": lr,
        "seed": seed,
        "loss_scales": LOSS_SCALES,
        "n_train_samples": len(samples),
        "history": history,
        "seconds": time.perf_counter() - started,
    }

adaptation = adapt_model(
    adapted_model,
    train_windows,
    val_window,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    mode=TRAINABLE,
    seed=SEED,
)

(OUTPUT_ROOT / "adaptation" / "training_history.json").write_text(
    json.dumps(adaptation, indent=2), encoding="utf-8"
)

display(pd.DataFrame(adaptation["history"]))
print({
    "best_epoch": adaptation["best_epoch"],
    "trainable_parameters": adaptation["n_trainable"],
    "percent_of_base": 100 * adaptation["n_trainable"] / PARAMETER_COUNT,
    "training_seconds": adaptation["seconds"],
})

# 13. Validate the selected adapted model

The training objective is one-step scaled MSE. The scientifically relevant question, however, is how the selected model behaves during an autoregressive rollout.

The next cell compares the selected adapted model with its own persistence baseline over 6–24 hours **on validation only**.

In [ ]:
# @title Adapted validation rollout
adapted_validation = evaluate_model(adapted_model, val_window, MAX_LEAD_STEPS)

(OUTPUT_ROOT / "adaptation" / "validation_metrics.json").write_text(
    json.dumps(adapted_validation, indent=2), encoding="utf-8"
)

val_compare = pd.DataFrame([
    {
        "lead": lead,
        "frozen_mean_ratio": frozen_validation["summary"][lead]["mean_skill"],
        "adapted_mean_ratio": adapted_validation["summary"][lead]["mean_skill"],
        "frozen_vars_beating_persistence": frozen_validation["summary"][lead]["variables_beating_persistence"],
        "adapted_vars_beating_persistence": adapted_validation["summary"][lead]["variables_beating_persistence"],
    }
    for lead in frozen_validation["leads"]
])
display(val_compare)

# 14. Freeze the experiment before the independent test

No test metric has been used for:

- choosing LoRA versus `lora+heads`;
- choosing the epoch;
- changing learning rate;
- changing lead times;
- changing the persistence definition; or
- changing the metric.

The next record freezes those choices before final test evaluation.

In [ ]:
# @title Freeze experiment
frozen_experiment = {
    "notebook_spec": "2.1",
    "profile": "E2E",
    "mode": "WORKSHOP",
    "model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "checkpoint_name": CHECKPOINT_NAME,
        "checkpoint_bytes": CHECKPOINT_BYTES,
        "checkpoint_sha256": CHECKPOINT_SHA256,
        "checkpoint_loading": "upstream weights_only checkpoint loader after local digest verification",
        "small_checkpoint_boundary": True,
    },
    "dataset": dataset_manifest,
    "state_contract": {
        "surface_variables": list(SURF_VARS),
        "atmospheric_variables": list(ATMOS_VARS),
        "static_variables": list(STATIC_VARS),
        "pressure_levels_hpa": list(LEVELS),
        "history_steps": HISTORY_STEPS,
        "timestep_hours": TIMESTEP_HOURS,
        "tutorial_resolution_degrees": val_window["resolution_degrees"],
        "training_resolution_degrees": 0.25,
    },
    "adaptation": {
        k: v for k, v in adaptation.items()
        if k not in {"trainable_names"}
    },
    "trainable_tensor_names": adaptation["trainable_names"],
    "evaluation": {
        "metric": "latitude-weighted RMSE with cosine-latitude weights normalized to unit mean",
        "persistence": "latest analysis carried unchanged to every lead",
        "skill_semantics": "model RMSE / persistence RMSE; below 1 beats persistence",
        "lead_hours": [6 * i for i in range(1, MAX_LEAD_STEPS+1)],
        "z500_is_diagnostic_not_tenth_summary_variable": True,
    },
    "validation": {
        "frozen": frozen_validation,
        "adapted": adapted_validation,
    },
}
freeze_path = OUTPUT_ROOT / "frozen" / "frozen_experiment.json"
freeze_path.write_text(
    json.dumps(frozen_experiment, indent=2, default=str),
    encoding="utf-8",
)
print("Experiment frozen:", freeze_path)

# 15. Independent October 2021 test

The test now compares three forecast systems:

1. persistence;
2. a **fresh** zero-LoRA Aurora model;
3. the validation-selected adapted model.

The frozen model is reconstructed only after the experiment is frozen, so its test metrics cannot influence the adaptation choices.

In [ ]:
# @title Independent test comparison
assert json.loads(freeze_path.read_text())["model"]["checkpoint_sha256"] == CHECKPOINT_SHA256

test_persistence = persistence_only(test_window, MAX_LEAD_STEPS)

fresh_frozen_model = build_aurora(use_lora=True)

started = time.perf_counter()
frozen_test = evaluate_model(fresh_frozen_model, test_window, MAX_LEAD_STEPS)
frozen_test_seconds = time.perf_counter() - started

started = time.perf_counter()
adapted_test = evaluate_model(adapted_model, test_window, MAX_LEAD_STEPS)
adapted_test_seconds = time.perf_counter() - started

comparison = pd.DataFrame([
    {
        "lead_hours": int(lead[:-1]),
        "frozen_mean_rmse_ratio": frozen_test["summary"][lead]["mean_skill"],
        "adapted_mean_rmse_ratio": adapted_test["summary"][lead]["mean_skill"],
        "frozen_vars_beating_persistence": frozen_test["summary"][lead]["variables_beating_persistence"],
        "adapted_vars_beating_persistence": adapted_test["summary"][lead]["variables_beating_persistence"],
        "variables": 9,
    }
    for lead in frozen_test["leads"]
])
display(comparison)

comparison.to_csv(OUTPUT_ROOT / "test" / "comparison.csv", index=False)
(OUTPUT_ROOT / "test" / "frozen_metrics.json").write_text(
    json.dumps(frozen_test, indent=2), encoding="utf-8"
)
(OUTPUT_ROOT / "test" / "adapted_metrics.json").write_text(
    json.dumps(adapted_test, indent=2), encoding="utf-8"
)

variable_rows = []
for name in REPORTED:
    for lead in frozen_test["leads"]:
        variable_rows.append({
            "variable": name,
            "lead": lead,
            "persistence_rmse": frozen_test["variables"][name][lead]["persistence"],
            "frozen_rmse": frozen_test["variables"][name][lead]["model"],
            "frozen_ratio": frozen_test["variables"][name][lead]["skill"],
            "adapted_rmse": adapted_test["variables"][name][lead]["model"],
            "adapted_ratio": adapted_test["variables"][name][lead]["skill"],
            "unit": UNITS.get("z" if name == "z500" else name, ""),
        })
variable_table = pd.DataFrame(variable_rows)
display(variable_table[variable_table["variable"].isin(["2t", "msl", "z500", "u"])])
variable_table.to_csv(OUTPUT_ROOT / "test" / "variable_metrics.csv", index=False)

## Reference expectation, not a canned answer

A previous clean DIMER T4 run at the same 1.5° tutorial setup observed frozen → adapted mean RMSE ratios of approximately:

- 6 h: 1.568 → 0.904
- 12 h: 1.328 → 0.849
- 18 h: 1.237 → 0.867
- 24 h: 1.294 → 0.969

Those values are **not injected into this notebook's results**. The table above is recomputed from this execution.

# 16. Lead-time error growth

Autoregressive forecasts feed predictions back into the next model step, so errors can accumulate with lead.

The following curves keep physical RMSE and relative-to-persistence behavior visible together.

In [ ]:
# @title Lead-time curves for headline variables
for variable in ("2t", "msl", "z500", "u"):
    block = variable_table[variable_table["variable"] == variable].copy()
    leads = block["lead"].str.replace("h", "", regex=False).astype(int)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(leads, block["persistence_rmse"], marker="o", label="persistence")
    ax.plot(leads, block["frozen_rmse"], marker="o", label="frozen Aurora")
    ax.plot(leads, block["adapted_rmse"], marker="o", label="adapted Aurora")
    ax.set_xlabel("Lead time [hours]")
    ax.set_ylabel(f"RMSE [{block['unit'].iloc[0]}]")
    ax.set_title(f"{variable}: error growth with lead time")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "figures" / f"lead_rmse_{variable}.png", dpi=150, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(comparison["lead_hours"], comparison["frozen_mean_rmse_ratio"], marker="o", label="frozen")
ax.plot(comparison["lead_hours"], comparison["adapted_mean_rmse_ratio"], marker="o", label="adapted")
ax.axhline(1.0, linestyle="--", label="persistence")
ax.set_xlabel("Lead time [hours]")
ax.set_ylabel("Mean RMSE ratio vs persistence")
ax.set_title("Nine-variable mean skill relative to persistence")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "figures" / "test_mean_skill.png", dpi=150, bbox_inches="tight")
plt.show()

# 17. Spatial forecast and error maps

A global mean can hide large regional differences.

For a deterministic October 2021 origin, compare the 24-hour:

- ERA5 analysis;
- persistence;
- frozen forecast;
- adapted forecast;
- frozen absolute error;
- adapted absolute error.

The images remain on the native **1.5° tutorial grid**. They are not visually interpolated to pretend that 0.25° detail exists.

In [ ]:
# @title Spatial 24-hour maps
MAP_ORIGIN = 1
MAP_STEPS = 4

frozen_preds = forecast_from_origin(fresh_frozen_model, test_window, MAP_ORIGIN, MAP_STEPS)
adapted_preds = forecast_from_origin(adapted_model, test_window, MAP_ORIGIN, MAP_STEPS)
frozen_24 = frozen_preds[-1]
adapted_24 = adapted_preds[-1]

height = frozen_24.surf_vars["2t"].shape[-2]
lat_map = test_window["lat"][:height]
lon_map = test_window["lon"]
extent = [lon_map[0], lon_map[-1], lat_map[-1], lat_map[0]]
truth_idx = MAP_ORIGIN + MAP_STEPS
z500_idx = LEVELS.index(500)

spatial_cases = {
    "2t": {
        "truth": test_window["surf"]["2t"][truth_idx, :height],
        "persistence": test_window["surf"]["2t"][MAP_ORIGIN, :height],
        "frozen": frozen_24.surf_vars["2t"][0,0].numpy(),
        "adapted": adapted_24.surf_vars["2t"][0,0].numpy(),
        "unit": "K",
    },
    "msl": {
        "truth": test_window["surf"]["msl"][truth_idx, :height],
        "persistence": test_window["surf"]["msl"][MAP_ORIGIN, :height],
        "frozen": frozen_24.surf_vars["msl"][0,0].numpy(),
        "adapted": adapted_24.surf_vars["msl"][0,0].numpy(),
        "unit": "Pa",
    },
    "z500": {
        "truth": test_window["atmos"]["z"][truth_idx, z500_idx, :height],
        "persistence": test_window["atmos"]["z"][MAP_ORIGIN, z500_idx, :height],
        "frozen": frozen_24.atmos_vars["z"][0,0,z500_idx].numpy(),
        "adapted": adapted_24.atmos_vars["z"][0,0,z500_idx].numpy(),
        "unit": "m²/s²",
    },
}

for name, case in spatial_cases.items():
    fields = [
        ("ERA5 analysis", case["truth"]),
        ("Persistence", case["persistence"]),
        ("Frozen Aurora", case["frozen"]),
        ("Adapted Aurora", case["adapted"]),
        ("Frozen |error|", np.abs(case["frozen"] - case["truth"])),
        ("Adapted |error|", np.abs(case["adapted"] - case["truth"])),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, (title, field) in zip(axes.ravel(), fields):
        im = ax.imshow(field, extent=extent, aspect="auto")
        ax.set_title(title)
        ax.set_xlabel("Lon")
        ax.set_ylabel("Lat")
        plt.colorbar(im, ax=ax, fraction=0.026)
    fig.suptitle(f"{name} at +24 h [{case['unit']}]")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "figures" / f"spatial_24h_{name}.png", dpi=140, bbox_inches="tight")
    plt.show()

# 18. New-origin forecast

The final two analyses of the test window are now treated as the initial condition for a forecast extending beyond the supplied reference period.

Because the future analyses are not part of the sample:

**evaluation status = not measurable yet**

In [ ]:
# @title Forecast 24 hours beyond the available window
NEW_ORIGIN = test_window["n_steps"] - 1
future_preds = forecast_from_origin(adapted_model, test_window, NEW_ORIGIN, MAX_LEAD_STEPS)

future_summary = []
for step, pred in enumerate(future_preds, start=1):
    future_summary.append({
        "lead_hours": step * 6,
        "valid_time": str(pred.metadata.time[0]),
        "mean_2t_K": float(pred.surf_vars["2t"][0,0].mean()),
        "mean_msl_Pa": float(pred.surf_vars["msl"][0,0].mean()),
        "z500_global_mean": float(pred.atmos_vars["z"][0,0,z500_idx].mean()),
        "evaluation_status": "not-measurable",
    })

future_df = pd.DataFrame(future_summary)
display(future_df)
future_df.to_csv(OUTPUT_ROOT / "future" / "forecast_summary.csv", index=False)

np.savez_compressed(
    OUTPUT_ROOT / "future" / "forecast_fields.npz",
    **{
        f"lead_{(i+1)*6}h_2t": p.surf_vars["2t"][0,0].numpy()
        for i, p in enumerate(future_preds)
    },
)

# 19. Export the LoRA adapter

The adapter contains only tensors permitted by the selected adaptation scope.

Canonical LoRA-only output is roughly 2 MB, far smaller than the 451 MB base checkpoint.

The manifest records the immutable base checkpoint identity so an adapter cannot silently float across a different Aurora base.

In [ ]:
# @title SafeTensors adapter export
from safetensors.torch import save_file, load_file

ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "aurora_lora"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
adapter_path = ARTIFACT_DIR / "adapter.safetensors"

wanted = set(adaptation["trainable_names"])
adapter_tensors = {
    k: v.detach().cpu().contiguous()
    for k, v in adapted_model.state_dict().items()
    if k in wanted
}
if sorted(adapter_tensors) != sorted(wanted):
    raise RuntimeError("adapter tensor scope mismatch")

save_file(adapter_tensors, str(adapter_path), metadata={"format": "pt"})

artifact_manifest = {
    "format": "org.valcorza.aurora-earth-system.adapter.v1",
    "format_version": "1.0",
    "base_model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "checkpoint_name": CHECKPOINT_NAME,
        "checkpoint_sha256": CHECKPOINT_SHA256,
    },
    "adapter": {
        "trainable": TRAINABLE,
        "n_tensors": len(adapter_tensors),
        "n_parameters": adaptation["n_trainable"],
        "epochs": EPOCHS,
        "best_epoch": adaptation["best_epoch"],
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "loss_scales": LOSS_SCALES,
    },
    "history": adaptation["history"],
    "tensors": sorted(adapter_tensors),
    "files": [{
        "path": "adapter.safetensors",
        "bytes": adapter_path.stat().st_size,
        "sha256": sha256_file(adapter_path),
    }],
}
(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2), encoding="utf-8"
)

print(artifact_manifest["files"][0])

# 20. Fresh-boundary reload verification

A valid artifact is more than a file that deserializes.

The workshop reconstructs a fresh LoRA-capable Aurora from the verified base checkpoint, loads exactly the declared adapter tensors, and repeats one fixed forecast.

The adapted and reloaded outputs must match within a strict numerical tolerance.

In [ ]:
# @title Fresh reload and parity
reloaded_model = build_aurora(use_lora=True)

manifest = json.loads((ARTIFACT_DIR / "manifest.json").read_text())
if manifest["base_model"]["revision"] != MODEL_REVISION:
    raise ValueError("adapter base revision mismatch")
if manifest["base_model"]["checkpoint_sha256"] != CHECKPOINT_SHA256:
    raise ValueError("adapter base digest mismatch")
file_entry = manifest["files"][0]
if sha256_file(adapter_path) != file_entry["sha256"] or adapter_path.stat().st_size != file_entry["bytes"]:
    raise ValueError("adapter digest/size mismatch")

loaded = load_file(str(adapter_path))
if sorted(loaded) != sorted(manifest["tensors"]):
    raise ValueError("adapter tensor manifest mismatch")

state = reloaded_model.state_dict()
for key, value in loaded.items():
    if tuple(value.shape) != tuple(state[key].shape):
        raise ValueError(f"{key}: adapter shape mismatch")
merged = dict(state)
merged.update({k: v.to(state[k].dtype) for k, v in loaded.items()})
reloaded_model.load_state_dict(merged, strict=True)
reloaded_model.eval()

VERIFY_ORIGIN = 1
VERIFY_STEPS = 1
a = forecast_from_origin(adapted_model, val_window, VERIFY_ORIGIN, VERIFY_STEPS)[0]
b = forecast_from_origin(reloaded_model, val_window, VERIFY_ORIGIN, VERIFY_STEPS)[0]

surf_diff = max(
    float((a.surf_vars[k] - b.surf_vars[k]).abs().max()) for k in SURF_VARS
)
atmos_diff = max(
    float((a.atmos_vars[k] - b.atmos_vars[k]).abs().max()) for k in ATMOS_VARS
)

RELOAD_TOLERANCE = 1e-6
if max(surf_diff, atmos_diff) > RELOAD_TOLERANCE:
    raise RuntimeError(
        f"fresh reload parity failed: surf={surf_diff}, atmos={atmos_diff}"
    )

reload_report = {
    "max_abs_surf_diff": surf_diff,
    "max_abs_atmos_diff": atmos_diff,
    "tolerance": RELOAD_TOLERANCE,
    "status": "PASS",
}
print(reload_report)

# 21. BYOD contract

Full BYOD E2E mode requires **four NetCDF windows**:

- two training windows;
- one validation window;
- one independent test window.

The accepted workshop schema uses:

### Coordinates

`time`, `latitude`, `longitude`, `level`

### Surface fields

`2t`, `10u`, `10v`, `msl`  
(or `surf_2t`, `surf_10u`, ...)

### Atmospheric fields

`t`, `u`, `v`, `q`, `z`  
(or `atmos_t`, ...)

### Static fields

`static_lsm`, `static_z`, `static_slt`

All four windows must use the same global grid and exact 13-level set.

### Data privacy

BYOD files are processed in the selected notebook runtime and are not sent to DIMER workers or APIs. A hosted notebook remains an external compute environment. Do not upload confidential, embargoed, proprietary, security-sensitive, or operational forecast data unless authorized.

# 22. Optional experiments

### A. LoRA versus LoRA + heads

Set:

`TRAINABLE = "lora+heads"`

and rerun from the top.

This changes the adaptation scope and must be selected on validation only. Do not compare several scopes on the test window and keep the best one.

### B. Longer rollouts

Aurora supports up to eight six-hour steps in this workshop contract:

**48 hours**

The built-in two-day windows provide little statistical evidence at the longest leads. Treat longer-rollout results as diagnostic.

### C. Latitude bands

A useful extension is to break RMSE into:

- 90–60°N
- 60–30°N
- 30°N–30°S
- 30–60°S
- 60–90°S

One two-day held-out window cannot establish regional forecast quality.

# 23. Interpretation and operational limits

### Small checkpoint

This workshop uses Aurora's small debugging/testing checkpoint, not the production-scale 1.3B model.

### Resolution shift

Every built-in metric is measured at **1.5°**, not the 0.25° training grid.

### ERA5 is the reference

ERA5 is a reanalysis produced by assimilating observations into a numerical weather model. Error here means error relative to ERA5, not error relative to direct observations.

### Deterministic forecast

This workflow provides no:

- ensemble spread;
- calibrated forecast uncertainty;
- probability of threshold exceedance; or
- confidence interval.

### One tiny held-out period

Two days from October 2021 cannot establish general forecast skill across:

- seasons;
- tropical cyclones;
- blocking events;
- regional extremes;
- different analysis products.

### Operational boundary

Do not use this workshop output alone to issue or inform:

- public weather warnings;
- aviation decisions;
- marine routing;
- emergency management;
- energy dispatch;
- agricultural advisories; or
- other consequential weather decisions.

An operational workflow requires representative verification, uncertainty treatment, monitoring, appropriate fallback procedures, and qualified meteorological review.

# 24. Workshop exercises

### Exercise 1 — persistence

Before viewing the neural forecast, which atmospheric fields should persistence handle well at six hours?

### Exercise 2 — resolution transfer

Why does changing from 0.25° to 1.5° alter the meaning of a 4×4 model token even though the input variables and units are unchanged?

### Exercise 3 — weighted RMSE

Why does cosine-latitude weighting matter for a regular lat/lon grid?

### Exercise 4 — adaptation

If one-step validation loss improves, must the 24-hour autoregressive forecast also improve? Why not?

### Exercise 5 — deployment

What additional verification would you require before using an adapted Earth-system model for a Philippine forecasting application?

## 25. Export provenance and report bundle

In [ ]:
# @title Final provenance export
import datetime
import shutil

runtime_report = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

experiment_manifest = {
    "notebook_spec": "2.1",
    "notebook_profile": "E2E",
    "notebook_mode": "WORKSHOP",
    "workshop_revision": "0.1.0-candidate",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "checkpoint": CHECKPOINT_NAME,
        "bytes": CHECKPOINT_BYTES,
        "sha256": CHECKPOINT_SHA256,
        "parameters": PARAMETER_COUNT,
        "small_debug_checkpoint": True,
    },
    "dataset": dataset_manifest,
    "adaptation": {
        k: v for k, v in adaptation.items()
        if k != "trainable_names"
    },
    "test": {
        "comparison": comparison.to_dict("records"),
        "frozen_seconds": frozen_test_seconds,
        "adapted_seconds": adapted_test_seconds,
    },
    "artifact": artifact_manifest,
    "reload": reload_report,
    "runtime": runtime_report,
    "standalone_contract": {
        "git_clone_required": False,
        "dimer_source_runtime_fetch_required": False,
        "dimer_worker_required": False,
        "credentials_required": False,
        "default_upload_required": False,
    },
    "candidate_release_gates": [
        "fresh T4 Run all qualification",
        "reconcile anonymous xarray WeatherBench2 acquisition with the live DIMER 57-object digest-pinned data path",
        "record peak VRAM and wall times for the exact released notebook",
    ],
    "evidence_scope": (
        "small Aurora checkpoint, real ERA5 via WeatherBench2 at out-of-distribution 1.5° resolution, "
        "two training windows, one validation window, one independent two-day test window"
    ),
}

(OUTPUT_ROOT / "provenance" / "experiment_manifest.json").write_text(
    json.dumps(experiment_manifest, indent=2, default=str),
    encoding="utf-8",
)
(OUTPUT_ROOT / "workshop_summary.json").write_text(
    json.dumps({
        "model": "Aurora 0.25° small pretrained",
        "test_comparison": comparison.to_dict("records"),
        "best_epoch": adaptation["best_epoch"],
        "trainable_parameters": adaptation["n_trainable"],
        "adapter_bytes": adapter_path.stat().st_size,
        "reload": reload_report,
    }, indent=2, default=str),
    encoding="utf-8",
)

bundle = shutil.make_archive(
    str(Path(OUTPUT_DIR).resolve()) + "_DIMER_Aurora_Weather_Workshop_Report",
    "zip",
    root_dir=Path(OUTPUT_DIR).resolve(),
)

print({
    "experiment_manifest": str(OUTPUT_ROOT / "provenance" / "experiment_manifest.json"),
    "report_bundle": bundle,
    "report_sha256": sha256_file(bundle),
})

# 26. Troubleshooting

| Symptom | Likely cause | Corrective action |
|---|---|---|
| CUDA unavailable | CPU runtime selected | switch to a T4-class GPU before running from the top |
| Aurora checkpoint digest mismatch | changed/incomplete model asset | remove cached checkpoint and retry; never bypass the pin |
| anonymous GCS/Zarr open fails | transient WeatherBench2/GCS or package issue | retry in a fresh runtime; preserve the same store and role chunks |
| wrong pressure levels | incompatible dataset | provide exactly the 13 Aurora levels |
| longitude count rejected | grid incompatible with patch size | use a global grid whose longitude count is divisible by four |
| latitude count rejected | patch geometry mismatch | use a global grid with H mod 4 in {0,1} |
| temperature/pressure range rejection | wrong units or corrupt data | correct source units rather than widening the validator |
| frozen model loses to persistence | 1.5° OOD setup | treat as an expected scientific result, not an execution failure |
| LoRA validation worsens | adaptation did not help | epoch 0 remains a valid selection |
| test improves after tuning on it | test leakage | restart; choose configuration only from training/validation |
| fresh reload mismatch | artifact/base mismatch or nondeterministic execution issue | stop and inspect model/artifact identity |
| BYOD path missing | full E2E role set incomplete | provide all four NetCDF role paths |

A clear refusal is preferable to silently changing the Earth-system forecasting problem.

# Glossary

| Term | Meaning |
|---|---|
| **Analysis** | Estimated atmospheric state used as model initial condition/reference |
| **Reanalysis** | Retrospective analysis produced with a fixed assimilation/model system, e.g. ERA5 |
| **Forecast origin** | Latest observed analysis from which a forecast starts |
| **Lead time** | Time between forecast origin and predicted valid time |
| **Persistence** | Baseline that carries the latest analysis forward unchanged |
| **Autoregressive rollout** | Feed each predicted state into the next forecast step |
| **Latitude-weighted RMSE** | RMSE weighted by cos(latitude) to account for grid-cell area |
| **z500** | Geopotential at 500 hPa, a common mid-tropospheric diagnostic |
| **LoRA** | Low-rank adaptation of selected model projections |
| **Domain shift** | Difference between pretraining and deployment distributions |
| **Resolution shift** | Change in physical meaning/scale of model grid cells |
| **ERA5** | ECMWF global atmospheric reanalysis used here as reference |
| **WeatherBench2** | Benchmark/data infrastructure distributing standardized weather datasets |
| **Sample-sanity evidence** | Evidence that a bounded tutorial workflow executes; not operational verification |

In [ ]:
# @title Run-all completion summary
completion = {
    "notebook_spec": "2.1",
    "profile": "E2E",
    "mode": "WORKSHOP",
    "model": "microsoft/aurora — AuroraSmallPretrained",
    "model_revision": MODEL_REVISION,
    "device": DEVICE,
    "dataset": dataset_manifest["source"],
    "tutorial_resolution_degrees": dataset_manifest["resolution_degrees"],
    "training_resolution_degrees": 0.25,
    "best_epoch": adaptation["best_epoch"],
    "trainable_parameters": adaptation["n_trainable"],
    "adapter_bytes": adapter_path.stat().st_size,
    "reload_status": reload_report["status"],
    "output_directory": str(OUTPUT_ROOT.resolve()),
    "release_status": "candidate",
}
display(pd.Series(completion, name="value").to_frame())

print(
    "Workshop complete. Interpret all metrics as sample-sanity evidence on the small Aurora checkpoint "
    "at an out-of-distribution 1.5° grid, not as operational or native-resolution Aurora skill."
)